# Grade experimental

Notebook que percorre **etapa a etapa** o módulo [`experimento.py`](../src/experimento.py):

**corpus → embeddings → compressão em lote → métricas → grade (ε × ρ × método) → agregação → CSV**

Cadeia de notebooks: [`extracao_embedding.ipynb`](extracao_embedding.ipynb) → [`compressao_tokens.ipynb`](compressao_tokens.ipynb) → [`avaliacao_metricas.ipynb`](avaliacao_metricas.ipynb) → **este notebook** → [`trabalho_final.ipynb`](trabalho_final.ipynb).

Pré-requisitos: $F$ e operadores em [`compressores.py`](../src/compressores.py); métricas em [`metricas.py`](../src/metricas.py). A grade completa (400 reviews) e validação vs artigo ficam em [`trabalho_final.ipynb`](trabalho_final.ipynb) ou `make experiment`.

**Tempo estimado (CPU, mini-corpus):** ~30 s embeddings + ~1 min grade = **~2 min**.

In [1]:
import os
import time

import numpy as np

import compressores as cp
import experimento as exp
from compressores import COMPRESSORES
from constantes import EPSILONS, MAX_TOKENS, ORCAMENTOS, PASTA_RESULTADOS, SEED
from experimento import (
    CAMPOS_CSV,
    _combos_por_metodo,
    _processar_amostras,
    agregar,
    carregar_dados,
    expandir_grade,
    pooling,
    registrar_ambiente,
    rodar_grade,
    salvar_csv,
)
from dados import dividir_indices_estratificado

N_POR_CLASSE_MINI = 20
N_MINI = 2 * N_POR_CLASSE_MINI
EPS_DEMO = 0.95
RHO_DEMO = 0.5

print("Diretório de trabalho:", os.getcwd())

Diretório de trabalho: /home/THIAGO.OUVERNEY/Projects/ALC/Trabalho_Final/svd-leverage-token-compression


## 1. Corpus + embeddings — `carregar_dados`

`carregar_dados` encapsula o pipeline de entrada do experimento:

1. `carregar_imdb` — corpus balanceado IMDb
2. loop `texto_para_embedding` — uma matriz $F^{(i)} \in \mathbb{R}^{T_i \times D}$ por review
3. retorna também `tempo_embedding_medio` (útil para custo de materialização)

Usamos **40 reviews** (`n_por_classe=20`) para manter o notebook rápido; o protocolo completo usa 400.

In [2]:
amostras, rotulos, textos, tempo_embed = carregar_dados(
    n_por_classe=N_POR_CLASSE_MINI,
    max_amostras=N_MINI,
    max_tokens=MAX_TOKENS,
)
idx_treino, idx_teste = dividir_indices_estratificado(rotulos, semente=SEED)

ts = [F.shape[0] for F in amostras]
print(f"Corpus mini: {len(amostras)} reviews ({N_POR_CLASSE_MINI}/classe)")
print(f"Treino: {len(idx_treino)} | Teste: {len(idx_teste)}")
print(f"T médio: {np.mean(ts):.1f}  (min={min(ts)}, max={max(ts)})")
print(f"D (embedding): {amostras[0].shape[1]}  (esperado 384)")
print(f"Tempo embedding médio: {tempo_embed:.3f} s/review")

/home/THIAGO.OUVERNEY/Projects/ALC/Trabalho_Final/svd-leverage-token-compression/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9860.17it/s]


Corpus mini: 40 reviews (20/classe)
Treino: 28 | Teste: 12
T médio: 203.7  (min=16, max=254)
D (embedding): 384  (esperado 384)
Tempo embedding médio: 0.189 s/review


## 2. Pooling downstream — `pooling`

Antes do classificador Ridge, cada $F_{\mathrm{pod}}$ vira um vetor de review:

1. **média** ao longo das linhas (tokens mantidos)
2. **normalização $L_2$** (norma unitária)

Mesma lógica usada dentro de `_processar_amostras`.

In [3]:
F0 = amostras[0]
_, F_pod0, _ = cp.comprimir_svd(F0, orcamento=RHO_DEMO, variancia_explicada=EPS_DEMO)

v_bruto = F_pod0.mean(axis=0)
v = pooling(F_pod0)

print(f"F_pod shape: {F_pod0.shape}")
print(f"Norma antes L2:  {np.linalg.norm(v_bruto):.4f}")
print(f"Norma após L2:   {np.linalg.norm(v):.4f}  (esperado ~1.0)")

F_pod shape: (127, 384)
Norma antes L2:  2.0592
Norma após L2:   1.0000  (esperado ~1.0)


## 3. Uma configuração — `_processar_amostras`

Núcleo do experimento: para um **método**, **ε** (`variancia_explicada`) e **ρ** (`orcamento`):

1. aplica o compressor em **todas** as reviews
2. pooling + Ridge no split treino/teste
3. agrega métricas algébricas e de custo

Fluxo interno:

```
carregar_dados → split → _processar_amostras (por método/ε/ρ)
                              ↓
                         pooling → Ridge
                              ↓
                    expandir_grade → agregar → salvar_csv
```

In [4]:
run_svd = _processar_amostras(
    amostras, rotulos, "svd", RHO_DEMO,
    idx_treino, idx_teste, semente=0,
    variancia_explicada=EPS_DEMO,
)

print(f"Config: svd | eps={EPS_DEMO} | rho={RHO_DEMO} | n={len(amostras)}")
for chave, valor in run_svd.items():
    if isinstance(valor, float):
        print(f"  {chave:25s}: {valor:.4f}")
    else:
        print(f"  {chave:25s}: {valor}")

Config: svd | eps=0.95 | rho=0.5 | n=40
  acuracia_downstream      : 0.6667
  energia_espectral        : 0.9511
  fidelidade_reconstrucao  : 0.7789
  compressao               : 0.4997
  t_medio                  : 203.6750
  t_linha_medio            : 101.9000
  k_medio                  : 48.8000
  custo_tempo_operador     : 0.0517


## 4. Comparar métodos — loop sobre `COMPRESSORES`

Mesmo protocolo, **ε** e **ρ** fixos; comparamos os quatro operadores no mini-corpus.

In [5]:
print(f"Comparação (eps={EPS_DEMO}, rho={RHO_DEMO}, n={len(amostras)}):\n")
print(f"{'Método':<14} {'Acc':>6} {'C':>6} {'E_k':>6} {'R_k':>6} {'k':>5}")
print("-" * 48)

for metodo in COMPRESSORES:
    run = _processar_amostras(
        amostras, rotulos, metodo, RHO_DEMO,
        idx_treino, idx_teste, semente=0,
        variancia_explicada=EPS_DEMO,
    )
    ek = run["energia_espectral"]
    rk = run["fidelidade_reconstrucao"]
    km = run["k_medio"]
    ek_s = f"{ek:.3f}" if not np.isnan(ek) else "  nan"
    rk_s = f"{rk:.3f}" if not np.isnan(rk) else "  nan"
    km_s = f"{km:.1f}" if not np.isnan(km) else "  nan"
    print(
        f"{metodo:<14} {run['acuracia_downstream']:6.3f} "
        f"{run['compressao']:6.3f} {ek_s:>6} {rk_s:>6} {km_s:>5}"
    )

Comparação (eps=0.95, rho=0.5, n=40):

Método            Acc      C    E_k    R_k     k
------------------------------------------------
full            0.667  0.000    nan    nan   nan
random          0.667  0.500    nan    nan   nan
svd             0.667  0.500  0.951  0.779  48.8
svd_energia     0.667  0.500  0.951  0.779  48.8


## 5. Combos por método — `_combos_por_metodo`

A grade completa varia **ε** ∈ {0.90, 0.95, 0.99} e **ρ** ∈ {0.50, 0.25, 0.125}. Para evitar runs redundantes:

| Método | Combos executados | Motivo |
|--------|-------------------|--------|
| `full` | 1 (ε₀, ρ₀) | não poda; ε e ρ não afetam o resultado |
| `random` | 3 (ε₀, ρ) | ε irrelevante; ρ define $T'$ |
| `svd`, `svd_energia` | 9 (ε × ρ) | ambos afetam $k$ e seleção |

Total: **1 + 3 + 9 + 9 = 22 execuções efetivas**.

In [6]:
total_runs = 0
for metodo in COMPRESSORES:
    combos = _combos_por_metodo(metodo)
    total_runs += len(combos)
    print(f"{metodo:12s}: {len(combos)} combos  {combos[:2]}{'...' if len(combos) > 2 else ''}")

print(f"\nTotal execuções efetivas: {total_runs}")
print(f"Grade expandida (CSV):    {len(EPSILONS) * len(ORCAMENTOS) * len(COMPRESSORES)} linhas")

full        : 1 combos  [(0.9, 0.5)]
random      : 3 combos  [(0.9, 0.5), (0.9, 0.25)]...
svd         : 9 combos  [(0.9, 0.5), (0.9, 0.25)]...
svd_energia : 9 combos  [(0.9, 0.5), (0.9, 0.25)]...

Total execuções efetivas: 22
Grade expandida (CSV):    36 linhas


## 6. Expansão da grade — `expandir_grade`

`expandir_grade` replica linhas de `full` e `random` em **todos** os ε, produzindo 36 linhas CSV uniformes (mesmo valor de métricas, ε/ρ distintos no cabeçalho). SVD já tem uma linha por combo.

In [7]:
linhas_parciais = []
for metodo in COMPRESSORES:
    for epsilon, orcamento in _combos_por_metodo(metodo):
        metricas_run = _processar_amostras(
            amostras, rotulos, metodo, orcamento,
            idx_treino, idx_teste, semente=0,
            variancia_explicada=epsilon,
        )
        linhas_parciais.append({
            "metodo": metodo,
            "epsilon": epsilon,
            "orcamento": orcamento,
            "semente": 0,
            "modo": "imdb",
            **metricas_run,
        })

linhas_expandidas = expandir_grade(linhas_parciais)
print(f"Antes expandir_grade: {len(linhas_parciais)} linhas (runs efetivos)")
print(f"Depois expandir_grade: {len(linhas_expandidas)} linhas (CSV)")

full_rows = [l for l in linhas_expandidas if l["metodo"] == "full"]
accs_full = {l["acuracia_downstream"] for l in full_rows}
print(f"\nfull: {len(full_rows)} linhas, {len(accs_full)} valor(es) distinto(s) de acurácia (esperado 1)")

Antes expandir_grade: 22 linhas (runs efetivos)
Depois expandir_grade: 36 linhas (CSV)

full: 9 linhas, 1 valor(es) distinto(s) de acurácia (esperado 1)


## 7. Grade mini — `rodar_grade`

`rodar_grade` orquestra o loop completo (split + combos + `expandir_grade`). Passamos `amostras`/`rotulos` já calculados para **não re-embedar**.

In [8]:
print(f"Executando rodar_grade ({len(amostras)} reviews, {len(COMPRESSORES)} métodos)...")
t0 = time.perf_counter()
linhas = rodar_grade(
    max_amostras=N_MINI,
    max_tokens=MAX_TOKENS,
    amostras=amostras,
    rotulos=rotulos,
)
elapsed = time.perf_counter() - t0

print(f"Grade concluída em {elapsed:.0f} s")
print(f"Linhas retornadas: {len(linhas)}  (esperado {len(EPSILONS) * len(ORCAMENTOS) * len(COMPRESSORES)})")
print(f"Campos por linha: {len(CAMPOS_CSV)}")

Executando rodar_grade (40 reviews, 4 métodos)...
Grade concluída em 17 s
Linhas retornadas: 36  (esperado 36)
Campos por linha: 13


## 8. Agregação — `agregar`

`agregar` agrupa linhas por `(método, ε, ρ)` e calcula médias (acurácia, compressão, $E_k$, etc.). Com uma semente por config, `acuracia_desvio` = 0.

In [9]:
agregado = agregar(linhas)

def tabela_acuracia(agregado, epsilon=EPS_DEMO):
    metodos_ordem = ["full", "random", "svd", "svd_energia"]
    cabecalho = f"{'Método':<14}" + "".join(f"rho={r:<6.3f}" for r in ORCAMENTOS)
    print(f"Acurácia downstream (eps={epsilon})")
    print(cabecalho)
    print("-" * len(cabecalho))
    for metodo in metodos_ordem:
        if metodo not in agregado:
            continue
        por_rho = {
            p["orcamento"]: p
            for p in agregado[metodo]
            if abs(p["epsilon"] - epsilon) < 1e-9
        }
        vals = [f"{por_rho[r]['acuracia_media']:.3f}" for r in ORCAMENTOS if r in por_rho]
        print(f"{metodo:<14}" + "".join(f"{v:<12}" for v in vals))

tabela_acuracia(agregado, epsilon=EPS_DEMO)
print(f"\nMétodos agregados: {sorted(agregado.keys())}")

Acurácia downstream (eps=0.95)
Método        rho=0.500 rho=0.250 rho=0.125 
--------------------------------------------
full          0.667       0.667       0.667       
random        0.667       0.750       0.667       
svd           0.667       0.583       0.417       
svd_energia   0.667       0.667       0.583       

Métodos agregados: ['full', 'random', 'svd', 'svd_energia']


## 9. Persistência — `salvar_csv` e `CAMPOS_CSV`

O CSV usa colunas fixas em `CAMPOS_CSV`. Gravamos uma cópia demo em `resultados/` (não sobrescreve `resultados.csv` da grade completa).

In [10]:
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
csv_demo = os.path.join(PASTA_RESULTADOS, "experimento_demo.csv")
salvar_csv(linhas, csv_demo)

print("CAMPOS_CSV:", ", ".join(CAMPOS_CSV))
print(f"\nArquivo: {csv_demo} ({len(linhas)} linhas)")
print("\nPrimeiras 3 linhas (metodo, epsilon, orcamento, acuracia_downstream, compressao):")
for linha in linhas[:3]:
    print(
        f"  {linha['metodo']:12s} eps={linha['epsilon']:.2f} rho={linha['orcamento']:.3f} "
        f"acc={linha['acuracia_downstream']:.3f} C={linha['compressao']:.3f}"
    )

CAMPOS_CSV: metodo, epsilon, orcamento, semente, modo, acuracia_downstream, energia_espectral, fidelidade_reconstrucao, compressao, t_medio, t_linha_medio, k_medio, custo_tempo_operador

Arquivo: /home/THIAGO.OUVERNEY/Projects/ALC/Trabalho_Final/svd-leverage-token-compression/resultados/experimento_demo.csv (36 linhas)

Primeiras 3 linhas (metodo, epsilon, orcamento, acuracia_downstream, compressao):
  full         eps=0.90 rho=0.500 acc=0.667 C=0.000
  full         eps=0.90 rho=0.250 acc=0.667 C=0.000
  full         eps=0.90 rho=0.125 acc=0.667 C=0.000


## 10. Ambiente — `registrar_ambiente`

Metadados de reprodutibilidade registrados ao final de `main()` (versões, plataforma, parâmetros).

In [11]:
ambiente = registrar_ambiente()
ambiente["max_amostras"] = N_MINI
ambiente["max_tokens"] = MAX_TOKENS

for chave, valor in ambiente.items():
    print(f"  {chave}: {valor}")

  dataset: imdb
  python: 3.12.3
  numpy: 2.5.0
  plataforma: Linux-6.17.0-35-generic-x86_64-with-glibc2.39
  processador: x86_64
  max_amostras: 40
  max_tokens: 256


## 11. Próximo passo

Este notebook usou **40 reviews** para ilustrar o módulo. Para a grade completa e validação vs artigo:

- [`trabalho_final.ipynb`](trabalho_final.ipynb) — 400 reviews, tabelas, figura trade-off
- `make experiment` — CLI com saídas em `resultados/`

Figuras geradas pelo CLI: `tradeoff_acerto_compressao.png`, `epsilon_k_cortes.png` (via [`visualizacao.py`](../src/visualizacao.py)).